## setup

In [1]:
from typing import List, Callable, Any, Dict
from pydantic import BaseModel, Field

import os, requests, json, gc, rich, random
import pandas as pd
import numpy as np

from tqdm import tqdm

import faiss

from openai import OpenAI
from openai.types.chat import ParsedChatCompletion

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
OPENAI_CHAT_MODEL = "gpt-5-mini"

OPENAI_INPUT_COST = 0.025 * 1e-6
OPENAI_OUTPUT_COST = 2.000 * 1e-6

FCLIP_API_TOKEN = os.getenv("HF_API_TOKEN")
FCLIP_API_ENDPOINT = "https://precove-fclip-back3.hf.space/encode_texts"

BATCH_SIZE = 128
LOAD_FAISS_INDEX = True

In [3]:
os.makedirs("results", exist_ok=True)

In [4]:
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## models

In [5]:
class Product(BaseModel):
    title: str
    description: str

    @classmethod
    def from_row(cls, row: pd.Series) -> "Product":
        return cls(
            title=row["originalTitle"],
            description=row["longDescription"]
        )


class ScoredProduct(Product):
    score: float

In [6]:
class ScoringEval(BaseModel):
    reasoning: str = Field(
        description=(
            "STRICT MAXIMUM 30 WORDS. "
            "Explain exactly why this score was assigned. "
            "Be direct and objective."
        )
    )
    score: int = Field(
        description="Relevance score from 1 to 5.",
        ge=1, le=5
    )


class ScoringEvalResult(BaseModel):
    query: str
    product: ScoredProduct
    response: ScoringEval

In [7]:
class PairwiseEval(BaseModel):
    reasoning: str = Field(
        description=(
            "STRICT MAXIMUM 40 WORDS. "
            "State the deciding factor between the two products. "
            "If it is a tie, explicitly state whether both products are highly relevant, "
            "or both are completely irrelevant. Do not write an essay."
        )
    )
    best_product_index: int = Field(
        description=(
            "Returns the evaluation outcome:\n"
            " 0 : Product 1 is clearly better.\n"
            " 1 : Product 2 is clearly better.\n"
            " 2 : Tie (Both are GOOD) - Both products perfectly match the query.\n"
            "-1 : Tie (Both are BAD) - Both products are irrelevant or fail completely."
        ),
        ge=-1, le=2
    )
    

class PairwiseEvalResult(BaseModel):
    query: str
    product_openai: ScoredProduct
    product_fclip: ScoredProduct
    response: PairwiseEval

In [8]:
class PairwiseMetrics(BaseModel):
    wins: int = Field(ge=0)
    losses: int = Field(ge=0)
    good_ties: int = Field(ge=0)
    bad_ties: int = Field(ge=0)

    @property
    def total(self) -> int:
        return self.wins + self.losses + self.good_ties + self.bad_ties

    @property
    def valid_total(self) -> int:
        return self.wins + self.losses + self.good_ties

    @property
    def win_rate(self) -> float:
        return self.wins / self.total if self.total > 0 else 0.0

    @property
    def loss_rate(self) -> float:
        return self.losses / self.total if self.total > 0 else 0.0

    @property
    def good_tie_rate(self) -> float:
        return self.good_ties / self.total if self.total > 0 else 0.0

    @property
    def bad_tie_rate(self) -> float:
        return self.bad_ties / self.total if self.total > 0 else 0.0

    @property
    def net_win_rate(self) -> float:
        return (self.wins - self.losses) / self.valid_total if self.valid_total > 0 else 0.0

    @property
    def success_rate(self) -> float:
        return self.valid_total / self.total if self.total > 0 else 0.0

## utils

In [9]:
def load_json(path: str) -> Any:
    with open(path, "r") as f:
        return json.load(f)

In [10]:
def save_json(data: Any, path: str) -> None:
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

In [11]:
def create_embeddings_openai(texts: List[str]) -> List[List[float]]:
    response = client_openai.embeddings.create(
        input=texts,
        model=OPENAI_EMBEDDING_MODEL,
    )
    
    return [item.embedding for item in response.data]

In [12]:
def create_embeddings_fclip(texts: List[str]) -> List[List[float]]:
    headers = {
        "Authorization": f"Bearer {FCLIP_API_TOKEN}",
        "Content-Type": "application/json",
    }
    
    payload = json.dumps({"texts": texts})

    response = requests.request(
        method="POST",
        url=FCLIP_API_ENDPOINT,
        headers=headers,
        data=payload,
    )

    if response.ok:
        return response.json()["embeddings"]

    return None

In [13]:
def create_np_embedding(text: str, embed_func: Callable) -> np.ndarray:
    embedding = embed_func([text])[0]
    np_embedding = np.array(embedding, dtype=np.float32).reshape(1, -1)
    faiss.normalize_L2(np_embedding)

    return np_embedding

In [14]:
def create_np_embeddings(texts: List[str], embed_func: Callable, batch_size: int) -> np.ndarray:
    n, n_success, all_embeddings = 0, 0, []
    loop = tqdm(iterable=range(0, len(texts), batch_size))

    for i in loop:
        n += 1

        try:
            batch = texts[i : i + batch_size]
            batch_embeddings = embed_func(batch)

            if batch_embeddings is not None:
                all_embeddings.extend(batch_embeddings)
                n_success += 1
        
        except Exception as e:
            loop.set_description(str(e))
        
        success_rate = n_success / n
        loop.set_description(f"{success_rate=:.2f}")

    embeddings = np.array(all_embeddings, dtype=np.float32)

    gc.collect()
    del all_embeddings

    return embeddings

In [15]:
def search(
    embedding: np.ndarray,
    index: faiss.IndexFlatIP,
    dataset: List[Product],
    top_k: int
) -> List[ScoredProduct]:
    scores, indices = index.search(embedding, k=top_k)
    results = []

    for score, idx in zip(scores[0], indices[0]):
        product = dataset[idx]

        product = ScoredProduct(
            title=product.title,
            description=product.description,
            score=score
        )
        
        results.append(product)

    return results

In [16]:
def display_search_results(results: List[ScoredProduct]) -> None:
    for rank, product in enumerate(results):
        msg = (
            f"Rank: {rank}\n"
            f"Title: {product.title}\n"
            f"Description: {product.description}\n"
            f"Score: {product.score:.3f}\n"
        )

        rich.print(msg)

In [17]:
def calculate_pairwise_metrics(results: List['PairwiseEvalResult']) -> PairwiseMetrics:
    wins, losses, good_ties, bad_ties = 0, 0, 0, 0

    for result in results:
        idx = result.response.best_product_index
        
        if idx == 0:
            wins += 1
        elif idx == 1:
            losses += 1
        elif idx == 2:
            good_ties += 1
        elif idx == -1:
            bad_ties += 1

    return PairwiseMetrics(
        wins=wins,
        losses=losses,
        good_ties=good_ties,
        bad_ties=bad_ties
    )

In [45]:
def display_metrics(metrics: PairwiseMetrics):
    print(f"Total Comparisons : {metrics.total}")
    print(f"Valid Comparisons : {metrics.valid_total}")
    print("-" * 35)
    print(f"Wins              : {metrics.wins} ({metrics.win_rate:.1%})")
    print(f"Losses            : {metrics.losses} ({metrics.loss_rate:.1%})")
    print(f"Good Ties         : {metrics.good_ties} ({metrics.good_tie_rate:.1%})")
    print(f"Bad Ties          : {metrics.bad_ties} ({metrics.bad_tie_rate:.1%})")
    print("-" * 35)
    print(f"Net Win Rate      : {metrics.net_win_rate:.1%}")
    print(f"Success Rate      : {metrics.success_rate:.1%}")

## prompts

### system

In [18]:
SYSTEM_PROMPT_SCORING = """You are an expert e-commerce search quality evaluator.
Your task is to judge how relevant a retrieved product is to a user's search query.

# Grading Scale (1-5):
1 - Completely Irrelevant: The product has nothing to do with the query (e.g., query asks for shoes, product is a hat).
2 - Poor Match: Shares a vague category but misses crucial user constraints like gender, specific style, or color.
3 - Fair Match: The product is the right broad type, but misses a secondary but important detail requested by the user.
4 - Good Match: Highly relevant. Fits the core intent and most attributes. Might have a very minor mismatch (e.g., slightly different brand or shade).
5 - Perfect Match: Exact intent. Matches all explicit constraints (type, color, style, brand, usage) flawlessly.

Analyze the query and the product, write a brief reasoning, and then assign the score."""


In [19]:
PAIRWISE_COMPARISON_SYSTEM_PROMPT = """You are an expert e-commerce search quality evaluator.
Your task is to compare two retrieved products against a user's search query and determine which one is the more relevant match.

# Evaluation Criteria:
- Core Intent: Which product better matches the user's primary need?
- Attributes & Constraints: Which product adheres strictly to secondary details (color, brand, style, gender, usage)?
- Specificity: If both are relevant, which one matches the descriptive nuances more precisely?

# Instructions:
1. Briefly analyze how Product 1 matches the query.
2. Briefly analyze how Product 2 matches the query.
3. Compare their strengths and weaknesses relative to the exact query constraints.
4. Assign the final index based on these exact rules:
   -  0 : Product 1 is strictly more relevant.
   -  1 : Product 2 is strictly more relevant.
   -  2 : Tie (Both are GOOD) - Both products perfectly match the query and are highly relevant.
   - -1 : Tie (Both are BAD) - Both products fail to match the core intent and are completely irrelevant."""

### user

In [20]:
def create_user_prompt_scoring(query: str, product: Product) -> str:
    return f"""
User Query: "{query}"

Retrieved Product Title: "{product.title}"
Retrieved Product Description: "{product.description}"

Evaluate the relevance."""

In [21]:
def create_user_prompt_pairwise(
    query: str, 
    product_1: Product, 
    product_2: Product
) -> str:
    return f"""
User Query: "{query}"

--- PRODUCT 1 ---
Title: "{product_1.title}"
Description: "{product_1.description}"

--- PRODUCT 2 ---
Title: "{product_2.title}"
Description: "{product_2.description}"

Compare the two products and output the index of the best match."""

## agents

In [22]:
class OpenAIAgent:
    def __init__(
        self, 
        system_prompt: str,
        output_schema: BaseModel,
        model: str = OPENAI_CHAT_MODEL, 
        temperature: float = 1.
    ):
        self.system_prompt = system_prompt
        self.output_schema = output_schema
        self.model = model
        self.temperature = temperature

        self._response = None
        self._total_prompt_tokens, self._total_completion_tokens = 0, 0
        self._last_prompt_tokens, self._last_completion_tokens = 0, 0

    @property
    def total_prompt_tokens(self) -> int:
        return self._total_prompt_tokens

    @property
    def total_completion_tokens(self) -> int:
        return self._total_completion_tokens

    @property
    def total_tokens(self) -> int:
        return self._total_prompt_tokens + self._total_completion_tokens

    @property
    def last_tokens(self) -> int:
        return self._last_prompt_tokens + self._last_completion_tokens

    @property
    def last_prompt_tokens(self) -> int:
        return self._last_prompt_tokens

    @property
    def last_completion_tokens(self) -> int:
        return self._last_completion_tokens
        
    def generate(self, text: str) -> ParsedChatCompletion:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": text}
        ]
        
        self._response = client_openai.beta.chat.completions.parse(
            model=self.model,
            messages=messages,
            response_format=self.output_schema,
            temperature=self.temperature
        )

        return self._response.choices[0].message.parsed

    def update_tokens(self):
        if self._response is not None:
            self._total_prompt_tokens += self._response.usage.prompt_tokens
            self._total_completion_tokens += self._response.usage.completion_tokens

            self._last_prompt_tokens = self._response.usage.prompt_tokens
            self._last_completion_tokens = self._response.usage.completion_tokens

## dataset

In [23]:
df = pd.read_csv("data/joko_products.csv")
print(df.shape)
df.head()

(17062, 2)


,originalTitle,longDescription
0,Light Blue Wash Fray Waistband Low Waist Strai...,Stay on trend with the light blue wash fray wa...
1,Dark Chocolate Cinched Long Sleeve Denim Jacket,Enhance your aesthetic in this dark chocolate ...
2,Black Diamante Detail Oversized Blazer Dress,We're all about the glam vibes this season and...
3,Petite Black Snatched Sculpt Strappy Maxi Dress,Master the minimalist mood with this black str...
4,Indigo Layered Exposed Pocket Wide Leg Jeans,"Consider these indigo, wide-leg jeans a master..."


In [24]:
products = [
    Product.from_row(row) for _, row in df.iterrows()
]

len(products)

17062

In [25]:
texts_openai = (
    df["originalTitle"].fillna("") + " " +
    df["longDescription"].fillna("")
).str.strip().tolist()

In [26]:
# we only use title since FashionCLIP is limited to 77 tokens
# and has been trained to map pixels to visual keywords that strictly describe clothing

texts_fclip = df["originalTitle"].fillna("").tolist()

## `FAISS` index

In [27]:
if LOAD_FAISS_INDEX:
    index_openai = faiss.read_index("data/index_openai.faiss")

else:
    embeddings_openai = create_np_embeddings(
        texts=texts_openai,
        embed_func=create_embeddings_openai,
        batch_size=BATCH_SIZE
    )
    
    faiss.normalize_L2(embeddings_openai)

    dim_openai = embeddings_openai.shape[1]
    index_openai = faiss.IndexFlatIP(dim_openai)
    index_openai.add(embeddings_openai)

    faiss.write_index(index_openai, "data/index_openai.faiss")

print(f"OpenAI index: {index_openai.ntotal} vectors, dim={index_openai.d}")

OpenAI index: 17062 vectors, dim=1536


In [28]:
if LOAD_FAISS_INDEX:
    index_fclip = faiss.read_index("data/index_fclip.faiss")

else:
    # takes some time to run since FashionCLIP encoder is hosted on Hugging Face CPU Basic space
    embeddings_fclip = create_np_embeddings(
        texts=texts_fclip,
        embed_func=create_embeddings_fclip,
        batch_size=BATCH_SIZE
    )

    faiss.normalize_L2(embeddings_fclip)

    dim_fclip = embeddings_fclip.shape[1]
    index_fclip = faiss.IndexFlatIP(dim_fclip)
    index_fclip.add(embeddings_fclip)

    faiss.write_index(index_fclip, "data/index_fclip.faiss")

print(f"FCLIP index: {index_fclip.ntotal} vectors, dim={index_fclip.d}")

FCLIP index: 17062 vectors, dim=512


## search

In [29]:
TOP_K = 3
query = "Ripped jeans with strass"

# "I want to take up yoga, what can I buy?"
# "Ripped jeans with strass"
# "Clothes for summer"
# "Outfit for attending a wedding"
# "Sports clothes for running"

In [30]:
embedding_openai = create_np_embedding(
    text=query, embed_func=create_embeddings_openai
)

results_openai = search(
    embedding=embedding_openai,
    index=index_openai,
    dataset=products,
    top_k=TOP_K
)

display_search_results(results_openai)

Rank: 0
Title: Washed Black Extreme Distressed Sequin Panel Straight Leg Jeans
Description: Turn heads for all the right reasons in these washed black extreme distressed sequin panel straight 
leg jeans. Brought to you in a washed black hue material with an extreme distressed detail and sequin panel design,
these straight leg jeans are a yes from us. Team with the matching top, buckle heels and simple accessories for a 
statement fit. Length approx 81cm/32" (Based on a sample size UK 8) Model wears size UK 8/ EU 36/ AUS 8/ US 4
Score: 0.519

Rank: 1
Title: Ecru Studded High Waist Jeans
Description: Keep it striking with the ecru studded high waist jeans. Made from an ecru denim material, they 
feature studded detailing, a high waist fit, and a straight leg cut. Style with a vest top, slip on mules and a 
clutch bag for timeless sophistication.
Score: 0.516

Rank: 2
Title: Washed Stone Frayed Striped Seam Wide Leg Jeans
Description: Introducing your new style staple with these washed stone frayed striped seam wide leg jeans. Brought 
to you in a washed stone material with a frayed design, striped seam detail and wide leg fit, these jeans are a 
must have. How can you resist Style with a white tee, fresh kicks and simple accessories for a look like no other. 
Length approx 78cm/30.5" (Based on a sample size UK 8) Model wears size UK 8/ EU 36/ AUS 8/ US 4
Score: 0.503

In [31]:
embedding_fclip = create_np_embedding(
    text=query, embed_func=create_embeddings_fclip
)

results_fclip = search(
    embedding=embedding_fclip,
    index=index_fclip,
    dataset=products,
    top_k=TOP_K
)

display_search_results(results_fclip)

Rank: 0
Title: Washed Stone Mid Rise Straight Leg Jeans
Description: Madre from a washed stone material with a must have low rise fit and a flattering straight leg design.
Take your look to new heights and pair these washed stone mid rise straight leg jeans with the matching long sleeve
jeacket and a pair of comfy sneakers to feel confident this season.
Score: 0.748

Rank: 1
Title: Washed Stone Low Rise Wide Leg Jeans
Description: Keep it casual with the washed stone low rise wide leg jeans. Featuring a relaxed fit, low-rise waist 
and soft washed finish, they re your new everyday essential. Style with a baby tee or cropped blazer for off-duty 
cool.
Score: 0.747

Rank: 2
Title: Washed Stone Mid Waist Wide Leg Jeans
Description: Define your denim aesthetic with a fresh perspective. The washed stone mid waist wide leg jeans offer 
a relaxed yet refined shape, featuring a clean mid-rise waist and an effortlessly flowing wide leg cut that feels 
directional. Classic denim details like front and back pockets, belt loops, and a button-fly fastening complete 
this essential style. Pair these jeans with a sharp blazer and tailored top for an elevated day look or a simple 
Vest and sneakers for off-duty cool.
Score: 0.744

## evaluation

In [32]:
agent_scoring = OpenAIAgent(
    system_prompt=SYSTEM_PROMPT_SCORING,
    output_schema=ScoringEval,
    model=OPENAI_CHAT_MODEL,
)

agent_pairwise = OpenAIAgent(
    system_prompt=PAIRWISE_COMPARISON_SYSTEM_PROMPT,
    output_schema=PairwiseEval,
    model=OPENAI_CHAT_MODEL,
)     

### single example

In [33]:
idx = 0
product_openai = results_openai[idx]
product_fclip = results_fclip[idx]

In [34]:
user_prompt_openai = create_user_prompt_scoring(
    query=query, product=product_openai
)

rich.print(user_prompt_openai)
print("-" * 100)

response_openai = agent_scoring.generate(text=user_prompt_openai)
agent_scoring.update_tokens()

print(f"Score: {response_openai.score}")
rich.print(response_openai.reasoning)
print(f"Tokens: {agent_scoring.last_tokens}")

User Query: "Ripped jeans with strass"

Retrieved Product Title: "Washed Black Extreme Distressed Sequin Panel Straight Leg Jeans"
Retrieved Product Description: "Turn heads for all the right reasons in these washed black extreme distressed 
sequin panel straight leg jeans. Brought to you in a washed black hue material with an extreme distressed detail 
and sequin panel design, these straight leg jeans are a yes from us. Team with the matching top, buckle heels and 
simple accessories for a statement fit. Length approx 81cm/32" (Based on a sample size UK 8) Model wears size UK 8/
EU 36/ AUS 8/ US 4"

Evaluate the relevance.

----------------------------------------------------------------------------------------------------
Score: 5


Product is distressed/ripped jeans with sparkly sequin panels, matching the 'ripped jeans with strass' intent; 
sequin embellishment serves same decorative purpose.

Tokens: 866


In [35]:
user_prompt_fclip = create_user_prompt_scoring(
    query=query, product=product_fclip
)

rich.print(user_prompt_fclip)
print("-" * 100)

response_fclip = agent_scoring.generate(text=user_prompt_fclip)
agent_scoring.update_tokens()

print(f"Score: {response_fclip.score}")
rich.print(response_fclip.reasoning)
print(f"Tokens: {agent_scoring.last_tokens}")

User Query: "Ripped jeans with strass"

Retrieved Product Title: "Washed Stone Mid Rise Straight Leg Jeans"
Retrieved Product Description: "Madre from a washed stone material with a must have low rise fit and a flattering 
straight leg design. Take your look to new heights and pair these washed stone mid rise straight leg jeans with the
matching long sleeve jeacket and a pair of comfy sneakers to feel confident this season."

Evaluate the relevance.

----------------------------------------------------------------------------------------------------
Score: 2


Is jeans but lacks rips and any strass/rhinestone embellishment; fails key user constraints.

Tokens: 673


In [36]:
user_prompt_pairwise = create_user_prompt_pairwise(
    query=query, 
    product_1=product_openai, 
    product_2=product_fclip
)

rich.print(user_prompt_fclip)
print("-" * 100)

response_pairwise = agent_pairwise.generate(text=user_prompt_pairwise)
agent_pairwise.update_tokens()

print(f"Best Product: {response_pairwise.best_product_index}")
rich.print(response_pairwise.reasoning)
print(f"Tokens: {agent_pairwise.last_tokens}")

User Query: "Ripped jeans with strass"

Retrieved Product Title: "Washed Stone Mid Rise Straight Leg Jeans"
Retrieved Product Description: "Madre from a washed stone material with a must have low rise fit and a flattering 
straight leg design. Take your look to new heights and pair these washed stone mid rise straight leg jeans with the
matching long sleeve jeacket and a pair of comfy sneakers to feel confident this season."

Evaluate the relevance.

----------------------------------------------------------------------------------------------------
Best Product: 0


Product 1 explicitly features extreme distressed (ripped) detailing and sequin panels (strass), matching both core 
intent and attributes. Product 2 lacks distressing and any embellishment.

Tokens: 946


### multiple examples

In [40]:
def run_pairwise_evaluation(
    queries: List[str],
    top_k: int,
    save_path: str | None = None,
) -> List[PairwiseEvalResult]:
    results: List[PairwiseEvalResult] = []
    results_json: List[Dict[str, Any]] = []
    n, n_success = 0, 0
    loop = tqdm(iterable=queries)

    for query in loop:
        n += top_k

        try:
            embeddings_openai = create_np_embedding(query, create_embeddings_openai)
            embeddings_fclip  = create_np_embedding(query, create_embeddings_fclip)

            products_openai = search(embeddings_openai, index_openai, products, top_k=top_k)
            products_fclip  = search(embeddings_fclip,  index_fclip,  products, top_k=top_k)

            for idx in range(top_k):
                product_openai = products_openai[idx]
                product_fclip  = products_fclip[idx]

                prompt = create_user_prompt_pairwise(
                    query=query,
                    product_1=product_openai,
                    product_2=product_fclip,
                )

                response: PairwiseEval = agent_pairwise.generate(text=prompt)
                agent_pairwise.update_tokens()

                eval_result = PairwiseEvalResult(
                    query=query,
                    product_openai=product_openai,
                    product_fclip=product_fclip,
                    response=response,
                )
            
                results.append(eval_result)
                n_success += 1

                if save_path is not None:
                    json_data = eval_result.model_dump()
                    results_json.append(json_data)
                    save_json(data=results_json, path=save_path)

        except Exception as e:
            loop.set_description(str(e))

        success_rate = n_success / n
        loop.set_description(f"{success_rate=:.2f}")

    return results

In [42]:
# OpenAI API costs increase with the number of retrieved products
TOP_K = 3

### simple queries

In [43]:
dataset_simple = load_json("queries/simple.json") 
queries = dataset_simple["queries"]

print(f"n={len(queries)}")
print(f"example: {random.choice(queries)}")

n=30
example: brown satin printed corset


In [44]:
results_simple = run_pairwise_evaluation(
    queries=dataset_simple["queries"],
    top_k=TOP_K,
    save_path="results/pairwise_simple.json"
)

success_rate=1.00: 100%|██████████| 30/30 [14:24<00:00, 28.82s/it]


In [46]:
metrics_simple = calculate_pairwise_metrics(results_simple)

display_metrics(metrics_simple)

Total Comparisons : 90
Valid Comparisons : 83
-----------------------------------
Wins              : 30 (33.3%)
Losses            : 25 (27.8%)
Good Ties         : 28 (31.1%)
Bad Ties          : 7 (7.8%)
-----------------------------------
Net Win Rate      : 6.0%
Success Rate      : 92.2%


### thematic queries

In [47]:
dataset_thematic = load_json("queries/thematic.json")
queries = dataset_thematic["queries"]

print(f"n={len(queries)}")
print(f"example: {random.choice(queries)}")

n=30
example: sophisticated evening wear


In [48]:
results_thematic = run_pairwise_evaluation(
    queries=dataset_thematic["queries"],
    top_k=TOP_K,
    save_path="results/pairwise_thematic.json"
)

success_rate=1.00: 100%|██████████| 30/30 [17:47<00:00, 35.58s/it]


In [49]:
metrics = calculate_pairwise_metrics(results_thematic)
display_metrics(metrics)

Total Comparisons : 90
Valid Comparisons : 90
-----------------------------------
Wins              : 66 (73.3%)
Losses            : 19 (21.1%)
Good Ties         : 5 (5.6%)
Bad Ties          : 0 (0.0%)
-----------------------------------
Net Win Rate      : 52.2%
Success Rate      : 100.0%


### specific queries

In [50]:
dataset_specific = load_json("queries/specific.json")  
queries = dataset_specific["queries"]

print(f"n={len(queries)}")
print(f"example: {random.choice(queries)}")

n=30
example: Petite Black Snatched Sculpt Strappy Maxi Dress


In [51]:
results_specific = run_pairwise_evaluation(
    queries=dataset_specific["queries"],
    top_k=TOP_K,
    save_path="results/pairwise_specific.json"
)

success_rate=1.00: 100%|██████████| 30/30 [13:50<00:00, 27.68s/it]


In [52]:
metrics = calculate_pairwise_metrics(results_specific)
display_metrics(metrics)

Total Comparisons : 90
Valid Comparisons : 77
-----------------------------------
Wins              : 19 (21.1%)
Losses            : 22 (24.4%)
Good Ties         : 36 (40.0%)
Bad Ties          : 13 (14.4%)
-----------------------------------
Net Win Rate      : -3.9%
Success Rate      : 85.6%


### costs

In [ ]:
input_tokens = agent_pairwise.total_prompt_tokens
output_tokens = agent_pairwise.total_completion_tokens

input_costs = input_tokens * OPENAI_INPUT_COST
output_costs = output_tokens * OPENAI_OUTPUT_COST
total_costs = input_costs + output_costs

print(f"OpenAI Costs: ${total_costs:.3f}")